# ДЗ 3. Завантаження даних та основи SQL (DQL): UCI Bank Marketing

**Датасет:** UCI Bank Marketing, файл `bank-additional-full.csv` (варіант C).
**Джерело:** https://archive.ics.uci.edu/dataset/222/bank+marketing (Moro, Cortez, Rita, 2014).
**Домен:** телемаркетингові кампанії португальського банку (травень 2008 – листопад 2010). Кожен рядок — це останній контакт із клієнтом у межах кампанії. Target `y` показує, чи оформив клієнт строковий депозит.
**Обсяг:** увесь файл, 41 188 рядків × 21 колонка. Це в межах рекомендованих 10–100 тис., тому вибірку не робимо.

Pipeline: офіційний ZIP → CSV у `data/` → `COPY FROM STDIN` у TEXT-staging → `INSERT INTO ... SELECT` з очищенням у типізовану таблицю → аудит якості та EDA **SQL-запитами в PostgreSQL**. Pandas використовується лише для відображення результатів.

Особливості датасету, які визначають рішення:
- CSV з роздільником `;`, значення в лапках, кінці рядків CRLF;
- `unknown` у категоріях, `999` у `pdays`, `nonexistent` у `poutcome` — три **різні** семантики, а не один «пропуск»;
- `duration` — тривалість дзвінка, відома лише **після** дзвінка, тобто leakage;
- рік контакту в файлі відсутній (є лише `month` і `day_of_week`).

## 1. Setup і завантаження

### 1.1. PostgreSQL через pgserver

In [ ]:
# Install dependencies. PyPI ships pgserver wheels only for CPython 3.9-3.12, but its C extension
# is built against the stable ABI (abi3), so on Python 3.13+ we download the cp312 wheel and
# retag it as cp312-abi3, which pip accepts for any CPython >= 3.12.
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(PIP + ["psycopg2-binary", "sqlalchemy", "pandas"], check=True)

if subprocess.run(PIP + ["pgserver"], capture_output=True).returncode != 0:
    wheel_dir = Path("/tmp/pgserver_wheel")
    subprocess.run(
        [sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
         "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
         "--platform", "manylinux2014_x86_64", "-d", str(wheel_dir)],
        check=True,
    )
    src = next(wheel_dir.glob("pgserver-0.1.4-cp312-cp312-*.whl"))
    dst = src.with_name(src.name.replace("cp312-cp312", "cp312-abi3"))
    src.rename(dst)
    subprocess.run(PIP + [str(dst)], check=True)

print(f"Python {sys.version.split()[0]}: залежності встановлено")

Python 3.13.16: залежності встановлено


In [ ]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

pg = pgserver.get_server("/tmp/hw3_pg", cleanup_mode="stop")
# Pin the psycopg2 driver explicitly: SQLAlchemy 2.1 maps "postgresql://" to psycopg 3,
# while COPY below uses psycopg2's cursor.copy_expert
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)


def q(sql: str) -> pd.DataFrame:
    # Run a SELECT and return the result as a DataFrame
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


def run_sql(sql: str) -> None:
    # Run a multi-statement SQL script in one transaction
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    conn.execute(text("SET TIME ZONE 'UTC'"))
print(version)

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### 1.2. Отримання даних з офіційного джерела

Офіційний архів UCI має вкладену структуру: `bank+marketing.zip` → `bank-additional.zip` → `bank-additional/bank-additional-full.csv`.
Код рекурсивно шукає потрібний файл у вкладених ZIP і пропускає службові `__MACOSX`. Сам CSV не змінюється, лише копіюється в `data/`.
Якщо файл уже є в `data/`, завантаження пропускається, тож повторний Run All не ходить у мережу.

In [ ]:
import hashlib
import io
import zipfile
from urllib.request import urlretrieve

DATA_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
TARGET_NAME = "bank-additional-full.csv"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
csv_path = DATA_DIR / TARGET_NAME


def find_in_zip(zf: zipfile.ZipFile, name: str) -> bytes | None:
    # Recursively search nested ZIP archives for a member with the given base name
    for member in zf.namelist():
        if member.startswith("__MACOSX") or member.endswith("/"):
            continue
        if Path(member).name == name:
            return zf.read(member)
        if member.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(zf.read(member))) as inner:
                found = find_in_zip(inner, name)
                if found is not None:
                    return found
    return None


if not csv_path.exists():
    zip_path, _ = urlretrieve(DATA_URL, "/tmp/bank_marketing.zip")
    with zipfile.ZipFile(zip_path) as zf:
        payload = find_in_zip(zf, TARGET_NAME)
    if payload is None:
        raise FileNotFoundError(f"{TARGET_NAME} не знайдено в архіві {DATA_URL}")
    csv_path.write_bytes(payload)

raw_bytes = csv_path.read_bytes()
print(f"Файл: {csv_path} ({len(raw_bytes) / 1e6:.2f} MB)")
print(f"MD5: {hashlib.md5(raw_bytes).hexdigest()}")
n_data_lines = raw_bytes.count(b"\n") - 1
print(f"Рядків даних (без header): {n_data_lines}")
print("Перші 2 рядки сирого файлу:")
for line in raw_bytes.splitlines()[:2]:
    print("  ", line.decode("utf-8"))

Файл: data/bank-additional-full.csv (5.83 MB)
MD5: f6cb2c1256ffe2836b36df321f46e92c
Рядків даних (без header): 41188
Перші 2 рядки сирого файлу:
   "age";"job";"marital";"education";"default";"housing";"loan";"contact";"month";"day_of_week";"duration";"campaign";"pdays";"previous";"poutcome";"emp.var.rate";"cons.price.idx";"cons.conf.idx";"euribor3m";"nr.employed";"y"
   56;"housemaid";"married";"basic.4y";"no";"no";"no";"telephone";"may";"mon";261;1;999;0;"nonexistent";1.1;93.994;-36.4;4.857;5191;"no"


### 1.3. Staging-таблиця і `COPY FROM STDIN`

Усі 21 колонки staging мають тип `TEXT`. Тому значення на кшталт `unknown`, `999` чи порожнього рядка не можуть зірвати імпорт: типізація відбувається пізніше й контрольовано.
Додаткова колонка `src_line_no` (identity) фіксує порядок рядків у файлі. Вона потрібна для lineage і для відновлення року контакту (розділ 2).
Колонки з крапкою в назві (`emp.var.rate`) перейменовано на snake_case. Порядок колонок у `COPY` збігається з порядком у header.

In [ ]:
STAGING_DDL = """
DROP TABLE IF EXISTS hw3_bank_contacts CASCADE;
DROP TABLE IF EXISTS hw3_bank_staging CASCADE;

CREATE TABLE hw3_bank_staging (
    src_line_no        BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    age_raw            TEXT,
    job_raw            TEXT,
    marital_raw        TEXT,
    education_raw      TEXT,
    default_raw        TEXT,
    housing_raw        TEXT,
    loan_raw           TEXT,
    contact_raw        TEXT,
    month_raw          TEXT,
    day_of_week_raw    TEXT,
    duration_raw       TEXT,
    campaign_raw       TEXT,
    pdays_raw          TEXT,
    previous_raw       TEXT,
    poutcome_raw       TEXT,
    emp_var_rate_raw   TEXT,
    cons_price_idx_raw TEXT,
    cons_conf_idx_raw  TEXT,
    euribor3m_raw      TEXT,
    nr_employed_raw    TEXT,
    y_raw              TEXT
);
"""
run_sql(STAGING_DDL)

COPY_SQL = """
COPY hw3_bank_staging (
    age_raw, job_raw, marital_raw, education_raw, default_raw, housing_raw, loan_raw,
    contact_raw, month_raw, day_of_week_raw, duration_raw, campaign_raw, pdays_raw,
    previous_raw, poutcome_raw, emp_var_rate_raw, cons_price_idx_raw, cons_conf_idx_raw,
    euribor3m_raw, nr_employed_raw, y_raw
)
FROM STDIN
WITH (FORMAT csv, HEADER true, DELIMITER ';', QUOTE '"')
"""

raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(COPY_SQL, file)
    raw_conn.commit()
finally:
    raw_conn.close()

q("SELECT COUNT(*) AS loaded_rows FROM hw3_bank_staging")

,loaded_rows
0,41188


In [ ]:
q("""
SELECT src_line_no, age_raw, job_raw, marital_raw, education_raw, default_raw,
       month_raw, day_of_week_raw, duration_raw, pdays_raw, poutcome_raw, euribor3m_raw, y_raw
FROM hw3_bank_staging
ORDER BY src_line_no
LIMIT 5
""")

,src_line_no,age_raw,job_raw,marital_raw,education_raw,default_raw,month_raw,day_of_week_raw,duration_raw,pdays_raw,poutcome_raw,euribor3m_raw,y_raw
0,1,56,housemaid,married,basic.4y,no,may,mon,261,999,nonexistent,4.857,no
1,2,57,services,married,high.school,unknown,may,mon,149,999,nonexistent,4.857,no
2,3,37,services,married,high.school,no,may,mon,226,999,nonexistent,4.857,no
3,4,40,admin.,married,basic.6y,no,may,mon,151,999,nonexistent,4.857,no
4,5,56,services,married,high.school,no,may,mon,307,999,nonexistent,4.857,no


**Висновок.** `COPY FROM STDIN` завантажив **41 188** рядків, стільки ж рядків даних у файлі (header пропущено через `HEADER true`). Роздільник `;`, лапки й кінці рядків CRLF розпізнано коректно: значення в staging без лапок, а `y_raw` без залишкового `\r`. Уже на сирих рядках видно три маркери, які обробляються по-різному: `unknown` у `default_raw`, `999` у `pdays_raw` та `nonexistent` у `poutcome_raw`.

## 2. Типізована таблиця та очищення

### 2.1. Семантика колонок, типи і правила очищення

Загальне правило для `unknown`: у цьому датасеті це **не бізнес-категорія**, а «значення не зібране» (клієнт не повідомив або банк не мав даних). Тому в clean-таблиці `unknown` стає `NULL`. Сам факт пропуску не втрачається: у SQL `NULL` відрізняється від будь-якого значення, а в feature engineering його можна закодувати як окрему категорію (розділ 6). Це рішення приймається свідомо й окремо для кожної колонки, а не як автоматична заміна.

Два інші «маркери» мають **іншу** семантику й обробляються інакше:
- `pdays = 999` означає «попереднього контакту не було». Це «не застосовується», а не «невідомо». Значення `999` не можна лишати числом, бо воно зламало б AVG чи MIN/MAX. Тому `days_since_prev_contact = NULL`, а факт контакту береться з `previous_contacts`.
- `poutcome = nonexistent` — **валідна категорія** «не було попередньої кампанії». Вона залишається категорією, а не `NULL`.

| Колонка (clean) | Сире поле | Семантика | Тип | Правило очищення |
|---|---|---|---|---|
| `contact_id` | — | технічний surrogate key | `BIGINT IDENTITY PK` | генерується |
| `src_line_no` | порядок у файлі | lineage до staging | `BIGINT NOT NULL UNIQUE` | з identity staging |
| `age` | `age` ("56") | вік клієнта, роки | `SMALLINT` + `CHECK 17..100` | `TRIM`→`::SMALLINT` |
| `job`, `marital`, `education` | `"admin."`, `"married"`, `"basic.4y"` | категорії клієнта, номінальні | `TEXT` + `CHECK IN (...)` | `TRIM`, `NULLIF(...,'unknown')`; бізнес-ліміту довжини немає, тому `TEXT`, а не `VARCHAR(255)` |
| `has_credit_default`, `has_housing_loan`, `has_personal_loan` | `"yes"/"no"/"unknown"` | справжні двійкові ознаки | `BOOLEAN` (nullable) | `yes→TRUE`, `no→FALSE`, `unknown→NULL` |
| `contact_channel` | `"cellular"/"telephone"` | канал дзвінка; дві категорії, але не «так/ні» | `TEXT` + `CHECK IN` | `TRIM` |
| `contact_month` | `month` ("may") + порядок рядків | календарний місяць контакту | `DATE` (1-ше число місяця) + `CHECK` | відновлення року, див. 2.2 |
| `contact_weekday` | `day_of_week` ("mon") | день тижня, лише робочі дні | `TEXT` + `CHECK IN ('mon'..'fri')` | `TRIM`, `LOWER` |
| `call_duration_sec` | `duration` ("261") | тривалість останнього дзвінка, с; **leakage** | `INTEGER` + `CHECK >= 0` | `::INTEGER` |
| `campaign_contacts` | `campaign` | кількість контактів у поточній кампанії (включно з цим) | `SMALLINT` + `CHECK >= 1` | `::SMALLINT` |
| `days_since_prev_contact` | `pdays` ("999") | днів з останнього контакту в попередній кампанії | `SMALLINT` nullable | `NULLIF(...,'999')` |
| `previous_contacts` | `previous` | контактів до цієї кампанії | `SMALLINT` + `CHECK >= 0` | `::SMALLINT` |
| `prev_outcome` | `poutcome` | результат попередньої кампанії | `TEXT NOT NULL` + `CHECK IN` | `nonexistent` залишається категорією |
| `emp_var_rate` | `emp.var.rate` ("1.1") | квартальна зміна зайнятості, % | `NUMERIC(3,1)` | точна десяткова, 1 знак у джерелі |
| `cons_price_idx` | `cons.price.idx` ("93.994") | щомісячний індекс споживчих цін | `NUMERIC(6,3)` | 3 знаки в джерелі |
| `cons_conf_idx` | `cons.conf.idx` ("-36.4") | щомісячний індекс довіри споживачів | `NUMERIC(4,1)` | може бути від'ємним |
| `euribor3m` | `euribor3m` ("4.857") | щоденна 3-місячна ставка Euribor, % | `NUMERIC(5,3)` + `CHECK >= 0` | ставка, 3 знаки |
| `nr_employed` | `nr.employed` ("5191") | квартальна кількість зайнятих, тис. | `NUMERIC(6,1)` | 1 знак у джерелі |
| `subscribed` | `y` ("no"/"yes") | **target**: оформив депозит | `BOOLEAN NOT NULL` | `yes→TRUE`, `no→FALSE`, інше → помилка |
| `loaded_at` | — | момент завантаження рядка (audit) | `TIMESTAMPTZ NOT NULL DEFAULT now()` | генерується |

Пояснення до вимог щодо типів:
- **`NUMERIC(p,s)` для грошей.** Грошових сум у `bank-additional-full.csv` немає: у варіанті `bank-full.csv` є `balance`, а тут ні. `NUMERIC(p,s)` використано для макроіндексів і ставки Euribor. Це точні десяткові значення з фіксованою точністю джерела, тому `FLOAT` тут не підходить.
- **`TIMESTAMPTZ`.** Точного моменту дзвінка в датасеті немає, і вигадувати його ми не будемо. `TIMESTAMPTZ` застосовано до справжньої події з конкретним моментом — завантаження рядка (`loaded_at`), це стандартна audit-колонка pipeline.
- **`DATE`.** `contact_month` — календарна дата без часу (перше число місяця). Це найточніша дата, яку дають дані.

Предметні CHECK-обмеження: діапазон віку; допустимі множини категорій; `campaign_contacts >= 1`; `call_duration_sec >= 0`; узгодженість `(prev_outcome = 'nonexistent') = (previous_contacts = 0)`; `days_since_prev_contact` заповнене лише за `previous_contacts > 0`; `contact_month` — перше число місяця в межах періоду кампаній.

### 2.2. DDL фінальної таблиці

In [ ]:
CLEAN_DDL = r"""
DROP TABLE IF EXISTS hw3_bank_contacts CASCADE;

CREATE TABLE hw3_bank_contacts (
    contact_id              BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    src_line_no             BIGINT NOT NULL UNIQUE,

    age                     SMALLINT NOT NULL
        CHECK (age BETWEEN 17 AND 100),
    job                     TEXT
        CHECK (job IN ('admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management',
                       'retired', 'self-employed', 'services', 'student', 'technician',
                       'unemployed')),
    marital                 TEXT
        CHECK (marital IN ('married', 'single', 'divorced')),
    education               TEXT
        CHECK (education IN ('basic.4y', 'basic.6y', 'basic.9y', 'high.school', 'illiterate',
                             'professional.course', 'university.degree')),
    has_credit_default      BOOLEAN,
    has_housing_loan        BOOLEAN,
    has_personal_loan       BOOLEAN,

    contact_channel         TEXT NOT NULL
        CHECK (contact_channel IN ('cellular', 'telephone')),
    contact_month           DATE NOT NULL
        CHECK (
            EXTRACT(DAY FROM contact_month) = 1
            AND contact_month BETWEEN DATE '2008-05-01' AND DATE '2010-11-01'
        ),
    contact_weekday         TEXT NOT NULL
        CHECK (contact_weekday IN ('mon', 'tue', 'wed', 'thu', 'fri')),
    call_duration_sec       INTEGER NOT NULL
        CHECK (call_duration_sec >= 0),
    campaign_contacts       SMALLINT NOT NULL
        CHECK (campaign_contacts >= 1),
    days_since_prev_contact SMALLINT
        CHECK (days_since_prev_contact >= 0),
    previous_contacts       SMALLINT NOT NULL
        CHECK (previous_contacts >= 0),
    prev_outcome            TEXT NOT NULL
        CHECK (prev_outcome IN ('nonexistent', 'failure', 'success')),

    emp_var_rate            NUMERIC(3, 1) NOT NULL,
    cons_price_idx          NUMERIC(6, 3) NOT NULL
        CHECK (cons_price_idx > 0),
    cons_conf_idx           NUMERIC(4, 1) NOT NULL,
    euribor3m               NUMERIC(5, 3) NOT NULL
        CHECK (euribor3m >= 0),
    nr_employed             NUMERIC(6, 1) NOT NULL
        CHECK (nr_employed > 0),

    subscribed              BOOLEAN NOT NULL,
    loaded_at               TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,

    -- "nonexistent" previous outcome <=> no contacts before this campaign
    CHECK ((prev_outcome = 'nonexistent') = (previous_contacts = 0)),
    -- days since previous contact only makes sense if there was a previous contact
    CHECK (days_since_prev_contact IS NULL OR previous_contacts > 0)
);
"""
run_sql(CLEAN_DDL)
print("hw3_bank_contacts створено")

hw3_bank_contacts створено


### 2.3. Очищення: `INSERT INTO ... SELECT`

**Відновлення року (`contact_month`).** У файлі немає року, але документація UCI каже, що рядки впорядковані за датою (травень 2008 – листопад 2010). Значить, номер місяця зростає в межах року й «падає» лише при переході на новий рік. Рік обчислюється як `2008 + кількість спадів номера місяця` від початку файлу (віконні `LAG` і кумулятивний `COUNT(*) FILTER`). Це **виведене** значення з явним припущенням, тому одразу після вставки воно перевіряється незалежно (2.4).

**Fail loudly.** Некоректне число чи дата (наприклад, `'abc'::SMALLINT`) зупинить `INSERT` помилкою, а не перетвориться мовчки на `NULL`. Для `subscribed` невідоме значення дає `NULL` і порушує `NOT NULL`. Тихі `NULL` можливі лише для `yes/no/unknown`-колонок, тому їхні сирі значення окремо перевіряються в аудиті 3.3.

**Фільтр.** Відкидаються лише рядки без ключових полів (вік, місяць, target). Конкретне правило — у `WHERE`.

In [ ]:
INSERT_SQL = r"""
WITH parsed AS (
    SELECT
        src_line_no,
        TRIM(age_raw)::SMALLINT                                        AS age,
        NULLIF(NULLIF(TRIM(job_raw), ''), 'unknown')                   AS job,
        NULLIF(NULLIF(TRIM(marital_raw), ''), 'unknown')               AS marital,
        NULLIF(NULLIF(TRIM(education_raw), ''), 'unknown')             AS education,
        CASE LOWER(TRIM(default_raw)) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE END
                                                                       AS has_credit_default,
        CASE LOWER(TRIM(housing_raw)) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE END
                                                                       AS has_housing_loan,
        CASE LOWER(TRIM(loan_raw))    WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE END
                                                                       AS has_personal_loan,
        LOWER(TRIM(contact_raw))                                       AS contact_channel,
        ARRAY_POSITION(
            ARRAY['jan', 'feb', 'mar', 'apr', 'may', 'jun',
                  'jul', 'aug', 'sep', 'oct', 'nov', 'dec'],
            LOWER(TRIM(month_raw))
        )                                                              AS month_num,
        LOWER(TRIM(day_of_week_raw))                                   AS contact_weekday,
        TRIM(duration_raw)::INTEGER                                    AS call_duration_sec,
        TRIM(campaign_raw)::SMALLINT                                   AS campaign_contacts,
        NULLIF(TRIM(pdays_raw), '999')::SMALLINT                       AS days_since_prev_contact,
        TRIM(previous_raw)::SMALLINT                                   AS previous_contacts,
        LOWER(TRIM(poutcome_raw))                                      AS prev_outcome,
        TRIM(emp_var_rate_raw)::NUMERIC(3, 1)                          AS emp_var_rate,
        TRIM(cons_price_idx_raw)::NUMERIC(6, 3)                        AS cons_price_idx,
        TRIM(cons_conf_idx_raw)::NUMERIC(4, 1)                         AS cons_conf_idx,
        TRIM(euribor3m_raw)::NUMERIC(5, 3)                             AS euribor3m,
        TRIM(nr_employed_raw)::NUMERIC(6, 1)                           AS nr_employed,
        CASE LOWER(TRIM(y_raw)) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE END
                                                                       AS subscribed
    FROM hw3_bank_staging
),
with_prev AS (
    SELECT
        p.*,
        LAG(month_num) OVER (ORDER BY src_line_no) AS prev_month_num
    FROM parsed AS p
),
with_year AS (
    SELECT
        w.*,
        2008 + COUNT(*) FILTER (WHERE month_num < prev_month_num)
                   OVER (ORDER BY src_line_no)    AS contact_year
    FROM with_prev AS w
)
INSERT INTO hw3_bank_contacts (
    src_line_no, age, job, marital, education,
    has_credit_default, has_housing_loan, has_personal_loan,
    contact_channel, contact_month, contact_weekday,
    call_duration_sec, campaign_contacts, days_since_prev_contact,
    previous_contacts, prev_outcome,
    emp_var_rate, cons_price_idx, cons_conf_idx, euribor3m, nr_employed,
    subscribed
)
SELECT
    src_line_no, age, job, marital, education,
    has_credit_default, has_housing_loan, has_personal_loan,
    contact_channel, MAKE_DATE(contact_year::INTEGER, month_num, 1), contact_weekday,
    call_duration_sec, campaign_contacts, days_since_prev_contact,
    previous_contacts, prev_outcome,
    emp_var_rate, cons_price_idx, cons_conf_idx, euribor3m, nr_employed,
    subscribed
FROM with_year
WHERE age IS NOT NULL
  AND month_num IS NOT NULL
  AND subscribed IS NOT NULL;
"""
run_sql(INSERT_SQL)
q("SELECT COUNT(*) AS clean_rows FROM hw3_bank_contacts")

,clean_rows
0,41188


### 2.4. Незалежна перевірка відновленого року

Якщо рік відновлено правильно, то **щомісячний** індекс споживчих цін `cons_price_idx` має рівно **одне** значення в кожному `contact_month`. Якщо ж групувати лише за назвою місяця без року, значень має бути кілька: «травень» тоді охоплює і 2008, і 2009, і 2010 рік.
Ця перевірка не використовує логіку, якою рік обчислено. Макроіндикатори — незалежне джерело.

In [ ]:
year_check = q("""
SELECT
    'contact_month (з роком)'                         AS grouping,
    COUNT(*)                                          AS n_groups,
    MAX(n_cpi)                                        AS max_distinct_cpi_per_group,
    MIN(first_month)                                  AS first_month,
    MAX(first_month)                                  AS last_month
FROM (
    SELECT contact_month AS first_month, COUNT(DISTINCT cons_price_idx) AS n_cpi
    FROM hw3_bank_contacts
    GROUP BY contact_month
) AS by_month
UNION ALL
SELECT
    'лише назва місяця (без року)',
    COUNT(*),
    MAX(n_cpi),
    NULL,
    NULL
FROM (
    SELECT EXTRACT(MONTH FROM contact_month) AS m, COUNT(DISTINCT cons_price_idx) AS n_cpi
    FROM hw3_bank_contacts
    GROUP BY EXTRACT(MONTH FROM contact_month)
) AS by_month_name
""")
year_check

,grouping,n_groups,max_distinct_cpi_per_group,first_month,last_month
0,contact_month (з роком),26,1,2008-05-01,2010-11-01
1,лише назва місяця (без року),10,3,None,None


In [ ]:
# Assertions: one CPI value per reconstructed month, period matches the documented May 2008 - Nov 2010
assert year_check.loc[0, "max_distinct_cpi_per_group"] == 1
assert str(year_check.loc[0, "first_month"]) == "2008-05-01"
assert str(year_check.loc[0, "last_month"]) == "2010-11-01"
print("Відновлення року підтверджено")

Відновлення року підтверджено


**Висновок.** Відновлений `contact_month` дає **26 місяців** з **рівно одним** значенням CPI в кожному. Групування лише за назвою місяця дає 10 груп і до **3** різних CPI в групі, бо змішуються роки. Період **2008-05 … 2010-11** збігається з документацією UCI. Отже, припущення про впорядкованість файлу підтверджене незалежним джерелом, і `contact_month` можна використовувати як `DATE`. Його обмеження: точність — місяць, а не день.

## 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [ ]:
counts = q("""
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_bank_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_bank_contacts
""")
counts

,layer,n_rows
0,staging,41188
1,clean,41188


**Висновок.** Кількості однакові: **41 188 = 41 188**. Правило фільтрації (`age`, `month_num` і `subscribed` не `NULL`) не відкинуло жодного рядка: усі ключові поля заповнені, назви місяців розпізнано, `y` має лише `yes/no`. Решта невалідних значень зупинила б `INSERT` помилкою приведення типу або порушенням `CHECK`, а не зникла б мовчки. Тому рівність кількостей означає, що всі рядки пройшли всі перевірки.

### 3.2. Перевірка дублікатів

Гарантованого природного ключа в датасеті **немає**: ідентифікатор клієнта чи дзвінка анонімізовано й видалено.
Тому за business key беремо **повний набір змістовних атрибутів контакту**: профіль клієнта, канал, місяць і день, тривалість, історію контактів, макроконтекст і результат. Технічні `contact_id`, `src_line_no`, `loaded_at` до нього не входять.
Перевірка виявляє лише **точні потенційні дублікати**. Два різні клієнти з однаковим профілем у принципі можливі, хоча збіг ще й тривалості дзвінка до секунди та всього контексту малоймовірний.

In [ ]:
dups = q("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        age, job, marital, education, has_credit_default, has_housing_loan, has_personal_loan,
        contact_channel, contact_month, contact_weekday, call_duration_sec, campaign_contacts,
        days_since_prev_contact, previous_contacts, prev_outcome,
        emp_var_rate, cons_price_idx, cons_conf_idx, euribor3m, nr_employed, subscribed
    )) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (
        age, job, marital, education, has_credit_default, has_housing_loan, has_personal_loan,
        contact_channel, contact_month, contact_weekday, call_duration_sec, campaign_contacts,
        days_since_prev_contact, previous_contacts, prev_outcome,
        emp_var_rate, cons_price_idx, cons_conf_idx, euribor3m, nr_employed, subscribed
    )) AS duplicate_rows
FROM hw3_bank_contacts
""")
dups

,total_rows,distinct_business_keys,duplicate_rows
0,41188,41176,12


Деталізація: які саме групи дублюються і чи стоять дублікати поруч у файлі.
`IS NOT DISTINCT FROM` у self-join потрібен, бо звичайне `=` не зіставило б рядки з однаковими `NULL` (наприклад, `job IS NULL`) і пропустило б такі дублікати.

In [ ]:
dup_pairs = q("""
SELECT
    a.src_line_no AS line_a,
    b.src_line_no AS line_b,
    b.src_line_no - a.src_line_no AS line_gap,
    a.age, a.job, a.contact_month, a.call_duration_sec, a.subscribed
FROM hw3_bank_contacts AS a
JOIN hw3_bank_contacts AS b
  ON a.src_line_no < b.src_line_no
 AND a.age = b.age
 AND a.call_duration_sec = b.call_duration_sec
 AND a.contact_month = b.contact_month
 AND a.job IS NOT DISTINCT FROM b.job
 AND a.marital IS NOT DISTINCT FROM b.marital
 AND a.education IS NOT DISTINCT FROM b.education
 AND a.has_credit_default IS NOT DISTINCT FROM b.has_credit_default
 AND a.has_housing_loan IS NOT DISTINCT FROM b.has_housing_loan
 AND a.has_personal_loan IS NOT DISTINCT FROM b.has_personal_loan
 AND a.contact_channel = b.contact_channel
 AND a.contact_weekday = b.contact_weekday
 AND a.campaign_contacts = b.campaign_contacts
 AND a.days_since_prev_contact IS NOT DISTINCT FROM b.days_since_prev_contact
 AND a.previous_contacts = b.previous_contacts
 AND a.prev_outcome = b.prev_outcome
 AND a.euribor3m = b.euribor3m
 AND a.nr_employed = b.nr_employed
 AND a.subscribed = b.subscribed
ORDER BY a.src_line_no
""")
dup_pairs

,line_a,line_b,line_gap,age,job,contact_month,call_duration_sec,subscribed
0,1266,1267,1,39,blue-collar,2008-05-01,124,False
1,12261,12262,1,36,retired,2008-07-01,88,False
2,14156,14235,79,27,technician,2008-07-01,331,False
3,16820,16957,137,47,technician,2008-07-01,43,False
4,18465,18466,1,32,technician,2008-07-01,128,False
5,20073,20217,144,55,services,2008-08-01,33,False
6,20532,20535,3,41,technician,2008-08-01,127,False
7,25184,25218,34,39,admin.,2008-11-01,123,False
8,28477,28478,1,24,services,2009-04-01,114,False
9,32506,32517,11,35,admin.,2009-05-01,348,False


**Висновок.** Знайдено **12 точних потенційних дублікатів** (0,03 %): 41 188 рядків, 41 176 унікальних. Кожен дублікат — пара. П'ять пар стоять у сусідніх рядках файлу (`line_gap = 1`), що схоже на подвійний запис однієї події, а не на двох різних клієнтів. 11 із 12 пар мають `subscribed = FALSE`.
**Дія:** у clean-таблиці рядки **залишено**. Без ID клієнта неможливо довести, що це той самий дзвінок, а clean-рівень не має мовчки змінювати дані джерела. У training data один рядок із кожної пари варто відкинути, бо обсяг мізерний, а при випадковому split дубль може опинитися і в train, і в test.

### 3.3. Null-rate і спеціальні маркери

**(а) Null-rate у clean-таблиці** — дев'ять колонок, включно з тими, де `NULL` неможливий за схемою (як контроль).

In [ ]:
nulls = q("""
SELECT
    ROUND(COUNT(*) FILTER (WHERE job IS NULL)                * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_job,
    ROUND(COUNT(*) FILTER (WHERE marital IS NULL)            * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_marital,
    ROUND(COUNT(*) FILTER (WHERE education IS NULL)          * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_education,
    ROUND(COUNT(*) FILTER (WHERE has_credit_default IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_default,
    ROUND(COUNT(*) FILTER (WHERE has_housing_loan IS NULL)   * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_housing,
    ROUND(COUNT(*) FILTER (WHERE has_personal_loan IS NULL)  * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_loan,
    ROUND(COUNT(*) FILTER (WHERE days_since_prev_contact IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_pdays,
    ROUND(COUNT(*) FILTER (WHERE age IS NULL)                * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_age,
    ROUND(COUNT(*) FILTER (WHERE subscribed IS NULL)         * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_target,
    COUNT(*) AS total_rows
FROM hw3_bank_contacts
""")
nulls

,pct_null_job,pct_null_marital,pct_null_education,pct_null_default,pct_null_housing,pct_null_loan,pct_null_pdays,pct_null_age,pct_null_target,total_rows
0,0.8,0.19,4.2,20.87,2.4,2.4,96.32,0.0,0.0,41188


**(б) Маркери пропусків у сирих даних.** Сканування всіх 21 staging-колонок на SQL `NULL`, порожній рядок, `unknown` та інші типові маркери (`?`, `NA`, `N/A`, `null`, `-`).
Колонки розгортаються в рядки через `CROSS JOIN LATERAL (VALUES ...)`, тож одна агрегація покриває всі поля.

In [ ]:
markers = q("""
SELECT
    col,
    COUNT(*) FILTER (WHERE val IS NULL)                          AS sql_null,
    COUNT(*) FILTER (WHERE TRIM(val) = '')                       AS empty_string,
    COUNT(*) FILTER (WHERE LOWER(TRIM(val)) = 'unknown')         AS unknown_marker,
    COUNT(*) FILTER (WHERE LOWER(TRIM(val)) IN ('?', 'na', 'n/a', 'null', '-')) AS other_markers
FROM hw3_bank_staging
CROSS JOIN LATERAL (
    VALUES
        ('age', age_raw), ('job', job_raw), ('marital', marital_raw),
        ('education', education_raw), ('default', default_raw), ('housing', housing_raw),
        ('loan', loan_raw), ('contact', contact_raw), ('month', month_raw),
        ('day_of_week', day_of_week_raw), ('duration', duration_raw),
        ('campaign', campaign_raw), ('pdays', pdays_raw), ('previous', previous_raw),
        ('poutcome', poutcome_raw), ('emp.var.rate', emp_var_rate_raw),
        ('cons.price.idx', cons_price_idx_raw), ('cons.conf.idx', cons_conf_idx_raw),
        ('euribor3m', euribor3m_raw), ('nr.employed', nr_employed_raw), ('y', y_raw)
) AS v (col, val)
GROUP BY col
HAVING COUNT(*) FILTER (WHERE val IS NULL OR TRIM(val) = ''
                          OR LOWER(TRIM(val)) IN ('unknown', '?', 'na', 'n/a', 'null', '-')) > 0
ORDER BY unknown_marker DESC
""")
markers

,col,sql_null,empty_string,unknown_marker,other_markers
0,default,0,0,8597,0
1,education,0,0,1731,0
2,loan,0,0,990,0
3,housing,0,0,990,0
4,job,0,0,330,0
5,marital,0,0,80,0


**(в) Семантичні маркери та контроль «тихих» NULL.** Окремо рахуємо `pdays = 999` і `poutcome = nonexistent`.
Також перевіряємо, що в `yes/no/unknown`-колонках немає інших сирих значень. Інакше `CASE` без `ELSE` тихо перетворив би їх на `NULL`.

In [ ]:
semantic = q("""
SELECT
    COUNT(*) FILTER (WHERE TRIM(pdays_raw) = '999')                     AS pdays_999,
    COUNT(*) FILTER (WHERE TRIM(poutcome_raw) = 'nonexistent')          AS poutcome_nonexistent,
    COUNT(*) FILTER (WHERE TRIM(previous_raw) = '0')                    AS previous_zero,
    COUNT(*) FILTER (
        WHERE LOWER(TRIM(default_raw)) NOT IN ('yes', 'no', 'unknown')
           OR LOWER(TRIM(housing_raw)) NOT IN ('yes', 'no', 'unknown')
           OR LOWER(TRIM(loan_raw))    NOT IN ('yes', 'no', 'unknown')
           OR LOWER(TRIM(y_raw))       NOT IN ('yes', 'no')
    )                                                                   AS unexpected_yes_no_values,
    COUNT(*) FILTER (
        WHERE LOWER(TRIM(housing_raw)) = 'unknown'
          AND LOWER(TRIM(loan_raw)) = 'unknown'
    )                                                                   AS housing_and_loan_both_unknown
FROM hw3_bank_staging
""")
semantic

,pdays_999,poutcome_nonexistent,previous_zero,unexpected_yes_no_values,housing_and_loan_both_unknown
0,39673,35563,35563,0,990


**Висновок.**
- **SQL `NULL` і порожніх рядків у сирому файлі немає.** Усі пропуски закодовано маркером `unknown`, причому лише в шести категоріальних колонках. Тому нульовий `sql_null` у staging не означає повноти даних, а null-rate у clean повністю відображає наше рішення `unknown → NULL`.
- **`has_credit_default` — 20,87 % `NULL`.** Невідомих значень (8 597) майже в 2 900 разів більше, ніж підтверджених дефолтів (3). Ознака фактично має вигляд «no / невідомо», і пропуск тут, імовірно, інформативний.
- **`has_housing_loan` і `has_personal_loan` — по 2,40 %, і це ті самі 990 рядків.** Пропуски систематичні: кредитна історія клієнта не була підтягнута цілком, а не випадкові.
- **`education` 4,20 %, `job` 0,80 %, `marital` 0,19 %.**
- **`days_since_prev_contact` — 96,32 % `NULL`.** Це не втрата даних, а «не застосовується»: 39 673 значення `999`. Їх **більше**, ніж `previous = 0` (35 563), на 4 110. Це суперечність, яку розібрано в 4.3.
- **`age` і `subscribed` — 0 %.** Це контроль `NOT NULL`, і він виконується.
- **Інших значень у `yes/no`-полях немає** (`unexpected_yes_no_values = 0`), тож `CASE` не створив тихих `NULL`. `nonexistent` = `previous = 0` = 35 563, отже ця категорія повністю узгоджена.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки: `job`

In [ ]:
dist_job = q("""
SELECT
    COALESCE(job, '(unknown → NULL)') AS job,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_bank_contacts
GROUP BY job
ORDER BY n_rows DESC, job
LIMIT 12
""")
dist_job

,job,n_rows,pct_rows
0,admin.,10422,25.30
1,blue-collar,9254,22.47
2,technician,6743,16.37
3,services,3969,9.64
4,management,2924,7.10
5,retired,1720,4.18
6,entrepreneur,1456,3.54
7,self-employed,1421,3.45
8,housemaid,1060,2.57
9,unemployed,1014,2.46


Для контексту — баланс target, який визначає вибір метрик:

In [ ]:
dist_target = q("""
SELECT
    subscribed,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_bank_contacts
GROUP BY subscribed
ORDER BY subscribed
""")
dist_target

,subscribed,n_rows,pct_rows
0,False,36548,88.73
1,True,4640,11.27


**Висновок.** Розподіл `job` сильно нерівномірний. Три категорії — `admin.` (25,3 %), `blue-collar` (22,5 %) і `technician` (16,4 %) — дають 64 % контактів. Рідкісні `student` (2,1 %), `unemployed` (2,5 %) та `unknown` (0,8 %) при one-hot кодуванні дадуть розріджені ознаки з нестабільними оцінками, тому їх можна об'єднати або застосувати target encoding з регуляризацією.
Target незбалансований: **11,27 %** `subscribed = TRUE` проти 88,73 %. Accuracy тут неінформативна (константа «ні» дає 88,7 %). Потрібні PR-AUC, recall/precision для класу «так» і стратифікований split.

### 4.2. Базова статистика числових колонок

Основна колонка — `campaign_contacts`; для порівняння форми розподілу додано `age` і `call_duration_sec`, плюс медіана та p99.

In [ ]:
stats = q("""
SELECT
    'campaign_contacts' AS metric,
    COUNT(campaign_contacts) AS n_non_null,
    MIN(campaign_contacts) AS min_value,
    MAX(campaign_contacts) AS max_value,
    ROUND(AVG(campaign_contacts)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(campaign_contacts)::NUMERIC, 2) AS sd_value,
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY campaign_contacts) AS median_value,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY campaign_contacts) AS p99_value
FROM hw3_bank_contacts
UNION ALL
SELECT
    'age', COUNT(age), MIN(age), MAX(age),
    ROUND(AVG(age)::NUMERIC, 2), ROUND(STDDEV_SAMP(age)::NUMERIC, 2),
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY age),
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY age)
FROM hw3_bank_contacts
UNION ALL
SELECT
    'call_duration_sec', COUNT(call_duration_sec), MIN(call_duration_sec), MAX(call_duration_sec),
    ROUND(AVG(call_duration_sec)::NUMERIC, 2), ROUND(STDDEV_SAMP(call_duration_sec)::NUMERIC, 2),
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY call_duration_sec),
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY call_duration_sec)
FROM hw3_bank_contacts
""")
stats

,metric,n_non_null,min_value,max_value,avg_value,sd_value,median_value,p99_value
0,campaign_contacts,41188,1,56,2.57,2.77,2.0,14.00
1,age,41188,17,98,40.02,10.42,38.0,71.00
2,call_duration_sec,41188,0,4918,258.29,259.28,180.0,1271.13


**Висновок.**
- **`campaign_contacts`** сильно скошений вправо: медіана 2, середнє 2,57, а максимум 56. P99 = 14 при стандартному відхиленні 2,77, тобто хвіст у кілька σ від центру. Для такого розподілу середнє і σ погано описують типового клієнта, краще дивитися на медіану та квантилі.
- **`age`** розподілений приблизно симетрично: середнє 40,0 ≈ медіана 38. Діапазон 17–98 правдоподібний, `CHECK` спрацював би на межах.
- **`call_duration_sec`** також має довгий хвіст: медіана 180 с, середнє 258, максимум 4 918 с (82 хв), p99 ≈ 1 271 с. Нуль присутній і перевіряється в 4.3.

### 4.3. Outlier-check

**Основний — доменний.** Перевіряємо порушення бізнес-логіки, а не просто «великі» значення:
1. `previous_contacts > 0`, але `pdays = 999` («не контактували»): поля суперечать одне одному;
2. `call_duration_sec = 0`: дзвінок фактично не відбувся;
3. `campaign_contacts > 30`: понад 30 дзвінків одному клієнту за кампанію;
4. `has_credit_default = TRUE`: рідкісна категорія.

In [ ]:
domain = q("""
SELECT
    COUNT(*) FILTER (WHERE previous_contacts > 0 AND days_since_prev_contact IS NULL) AS prev_contact_but_pdays_999,
    COUNT(*) FILTER (WHERE call_duration_sec = 0)                                    AS zero_duration_calls,
    COUNT(*) FILTER (WHERE call_duration_sec = 0 AND subscribed)                     AS zero_duration_subscribed,
    COUNT(*) FILTER (WHERE campaign_contacts > 30)                                   AS campaign_over_30,
    COUNT(*) FILTER (WHERE has_credit_default)                                       AS credit_default_yes
FROM hw3_bank_contacts
""")
domain

,prev_contact_but_pdays_999,zero_duration_calls,zero_duration_subscribed,campaign_over_30,credit_default_yes
0,4110,4,0,33,3


Як розподілено головну суперечність (1) за результатом попередньої кампанії:

In [ ]:
pdays_conflict = q("""
SELECT
    prev_outcome,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE days_since_prev_contact IS NULL) AS pdays_999,
    COUNT(*) FILTER (WHERE days_since_prev_contact IS NOT NULL) AS pdays_known
FROM hw3_bank_contacts
GROUP BY prev_outcome
ORDER BY n_rows DESC
""")
pdays_conflict

,prev_outcome,n_rows,pdays_999,pdays_known
0,nonexistent,35563,35563,0
1,failure,4252,4110,142
2,success,1373,0,1373


Приклади рядків для кожного правила (по 3):

In [ ]:
domain_rows = q("""
SELECT rule, src_line_no, age, job, campaign_contacts, previous_contacts,
       days_since_prev_contact, prev_outcome, call_duration_sec, subscribed
FROM (
    SELECT
        CASE
            WHEN previous_contacts > 0 AND days_since_prev_contact IS NULL THEN '1: previous>0, pdays=999'
            WHEN call_duration_sec = 0 THEN '2: zero duration'
            WHEN campaign_contacts > 30 THEN '3: campaign>30'
            WHEN has_credit_default THEN '4: default=yes'
        END AS rule,
        c.*,
        ROW_NUMBER() OVER (
            PARTITION BY CASE
                WHEN previous_contacts > 0 AND days_since_prev_contact IS NULL THEN 1
                WHEN call_duration_sec = 0 THEN 2
                WHEN campaign_contacts > 30 THEN 3
                WHEN has_credit_default THEN 4
            END
            ORDER BY src_line_no
        ) AS rn
    FROM hw3_bank_contacts AS c
    WHERE (previous_contacts > 0 AND days_since_prev_contact IS NULL)
       OR call_duration_sec = 0
       OR campaign_contacts > 30
       OR has_credit_default
) AS flagged
WHERE rn <= 3
ORDER BY rule, src_line_no
""")
domain_rows

,rule,src_line_no,age,job,campaign_contacts,previous_contacts,days_since_prev_contact,prev_outcome,call_duration_sec,subscribed
0,"1: previous>0, pdays=999",24014,38,blue-collar,1,1,None,failure,39,False
1,"1: previous>0, pdays=999",24020,40,services,1,1,None,failure,144,True
2,"1: previous>0, pdays=999",24077,36,admin.,1,1,None,failure,449,False
3,2: zero duration,6252,39,admin.,4,0,None,nonexistent,0,False
4,2: zero duration,23032,59,management,10,0,None,nonexistent,0,False
5,2: zero duration,28064,53,blue-collar,3,0,None,nonexistent,0,False
6,3: campaign>30,4108,32,admin.,56,0,None,nonexistent,261,False
7,3: campaign>30,4115,52,entrepreneur,39,0,None,nonexistent,44,False
8,3: campaign>30,4140,52,management,35,0,None,nonexistent,88,False
9,4: default=yes,21581,48,technician,1,0,None,nonexistent,66,False


**Додатково — статистичний 3σ для `campaign_contacts`.** Цей розподіл сильно скошений вправо, тож правило 3σ, розраховане на симетричний розподіл, тут лише ілюструє, чому сліпо застосовувати його не можна.

In [ ]:
zscore = q("""
WITH stats AS (
    SELECT
        AVG(campaign_contacts) AS mean_value,
        STDDEV_SAMP(campaign_contacts) AS sd_value
    FROM hw3_bank_contacts
    WHERE campaign_contacts IS NOT NULL
)
SELECT
    t.campaign_contacts,
    COUNT(*) AS n_rows,
    ROUND(((t.campaign_contacts - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score,
    ROUND(AVG(t.subscribed::INT) * 100, 1) AS pct_subscribed
FROM hw3_bank_contacts AS t
CROSS JOIN stats AS s
WHERE t.campaign_contacts IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.campaign_contacts - s.mean_value) > 3 * s.sd_value
GROUP BY t.campaign_contacts, s.mean_value, s.sd_value
ORDER BY t.campaign_contacts
LIMIT 20
""")
zscore

,campaign_contacts,n_rows,z_score,pct_subscribed
0,11,177,3.04,6.8
1,12,125,3.41,2.4
2,13,92,3.77,4.3
3,14,69,4.13,1.4
4,15,51,4.49,3.9
5,16,51,4.85,0.0
6,17,58,5.21,6.9
7,18,33,5.57,0.0
8,19,26,5.93,0.0
9,20,30,6.29,0.0


In [ ]:
q("""
WITH stats AS (
    SELECT AVG(campaign_contacts) AS m, STDDEV_SAMP(campaign_contacts) AS sd
    FROM hw3_bank_contacts
)
SELECT
    ROUND(m + 3 * sd, 2) AS threshold_3sigma,
    COUNT(*) FILTER (WHERE campaign_contacts > m + 3 * sd) AS n_flagged,
    ROUND(COUNT(*) FILTER (WHERE campaign_contacts > m + 3 * sd) * 100.0 / COUNT(*), 2) AS pct_flagged
FROM hw3_bank_contacts CROSS JOIN stats
GROUP BY m, sd
""")

,threshold_3sigma,n_flagged,pct_flagged
0,10.88,869,2.11


**Висновок за доменною перевіркою:**

| Правило | Знайдено | Пропонована дія |
|---|---|---|
| `previous > 0`, але `pdays = 999` | **4 110** (10 % датасету) | **Передати на перевірку джерела + прапорець.** Усі 4 110 мають `poutcome = failure`, тобто клієнта точно контактували раніше. Імовірно, `999` тут означає «давніше, ніж відстежується», а не «не було контакту». Тому факт контакту беремо з `previous_contacts`, а `days_since_prev_contact` для цих рядків вважаємо невідомим, а не «нескінченним». |
| `call_duration_sec = 0` | **4**, жоден не `subscribed` | **Залишити, позначити прапорцем.** Це дзвінки, які не відбулися. Для моделі до дзвінка `duration` і так не використовується. |
| `campaign_contacts > 30` | **33** | **Залишити як валідний рідкісний випадок.** Це реальна (хоч і сумнівна) практика повторних дзвінків, а не помилка вводу. |
| `has_credit_default = TRUE` | **3** | **Залишити.** Категорія надто рідкісна, щоб бути ознакою; фактично ознака бінарна «no / unknown». |

**Чому 3σ тут не підходить.** Поріг 10,88 позначає **869 рядків (2,11 %)**, тоді як для нормального розподілу очікувалося б ~0,13 %. Тобто правило ловить не помилки, а звичайний правий хвіст скошеного розподілу. Значення 11–56 правдоподібні, і конверсія в них стабільно низька (0–7 %). Це корисний сигнал для моделі, тож видаляти їх не можна. Для таких розподілів краще квантильний поріг або log-трансформація.

## 5. Десять DQL/EDA-запитів

Конверсія (`pct_subscribed`) — частка клієнтів, що оформили депозит. Базовий рівень по всьому набору наведено в 4.1. Таблиця покриття обов'язкових конструкцій — у кінці розділу.

### Запит 1

**Business-question:** Які категорії роботи мають найвищу конверсію серед груп з обсягом понад 500 контактів?

In [ ]:
q1 = q("""
SELECT
    COALESCE(job, '(unknown)') AS job,
    COUNT(*) AS n_contacts,
    COUNT(*) FILTER (WHERE subscribed) AS n_subscribed,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
GROUP BY job
HAVING COUNT(*) > 500
ORDER BY pct_subscribed DESC, job
LIMIT 5
""")
q1

,job,n_contacts,n_subscribed,pct_subscribed
0,student,875,275,31.43
1,retired,1720,434,25.23
2,unemployed,1014,144,14.20
3,admin.,10422,1352,12.97
4,management,2924,328,11.22


**Interpretation:** Найвищу конверсію серед великих груп мають **student (31,43 %)** і **retired (25,23 %)**, це в 2,8 і 2,2 раза вище за базові 11,27 %. Далі йдуть unemployed (14,20 %) і admin. (12,97 %) — найбільша група, з 1 352 підписками в абсолютних числах. Отже, «найконверсійніші» сегменти малі, а основний обсяг депозитів дає масовий сегмент admin. Для таргетингу важливі обидві метрики, а не лише частка.

### Запит 2

**Business-question:** Як конверсія змінюється залежно від вікової групи?

In [ ]:
q2 = q("""
SELECT
    CASE
        WHEN age BETWEEN 17 AND 24 THEN '17-24'
        WHEN age BETWEEN 25 AND 39 THEN '25-39'
        WHEN age BETWEEN 40 AND 59 THEN '40-59'
        ELSE '60+'
    END AS age_band,
    COUNT(*) AS n_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
GROUP BY age_band
ORDER BY age_band
""")
q2

,age_band,n_contacts,pct_subscribed
0,17-24,1068,23.97
1,25-39,21539,11.05
2,40-59,17388,8.80
3,60+,1193,39.56


**Interpretation:** Залежність від віку U-подібна. Конверсія найвища в крайніх групах: **60+ — 39,56 %** і **17–24 — 23,97 %**. Найнижча вона в 40–59 (8,80 %), а саме вікові групи 25–59 дають 95 % усіх контактів. Це узгоджується з Q1 (студенти та пенсіонери). Через нелінійність вік краще подавати моделі бінами або деревам, а не лінійним членом.

### Запит 3

**Business-question:** Чи відрізняється конверсія клієнтів з базовою освітою (basic.*) від клієнтів з вищою чи професійною освітою?

In [ ]:
q3 = q("""
SELECT
    CASE
        WHEN education LIKE 'basic.%' THEN 'basic (4y/6y/9y)'
        WHEN education ILIKE '%university%' OR education ILIKE '%professional%' THEN 'university / professional'
        ELSE COALESCE(education, '(unknown)')
    END AS education_group,
    COUNT(*) AS n_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
GROUP BY education_group
ORDER BY pct_subscribed DESC
""")
q3

,education_group,n_contacts,pct_subscribed
0,illiterate,18,22.22
1,(unknown),1731,14.50
2,university / professional,17411,13.01
3,high.school,9515,10.84
4,basic (4y/6y/9y),12513,8.70


**Interpretation:** Клієнти з базовою освітою (`basic.*`, 12 513 контактів) конвертуються найгірше — **8,70 %**. Університетська або професійна освіта дає **13,01 %**, high.school — 10,84 %. Група `illiterate` має 22,22 %, але це лише 18 рядків, тому висновок статистично ненадійний. `unknown` (14,50 %) конвертується **краще** за середнє, отже пропуск в освіті не випадковий і несе сигнал.

### Запит 4

**Business-question:** Яка частка клієнтів має невідомий кредитний статус (default, housing або personal loan) і чи конвертуються вони інакше?

In [ ]:
q4 = q("""
SELECT
    (has_credit_default IS NULL OR has_housing_loan IS NULL OR has_personal_loan IS NULL)
        AS any_credit_status_unknown,
    COUNT(*) AS n_contacts,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
GROUP BY any_credit_status_unknown
ORDER BY any_credit_status_unknown
""")
q4

,any_credit_status_unknown,n_contacts,pct_contacts,pct_subscribed
0,False,31828,77.27,12.88
1,True,9360,22.73,5.76


**Interpretation:** У **22,73 %** контактів (9 360) невідомий хоча б один кредитний статус. Головний внесок дає `default = unknown`. Конверсія в цій групі — **5,76 %**, удвічі нижча, ніж у групі з повністю відомими статусами (12,88 %). Отже, пропуск інформативний, і заміна `unknown` найчастішим значенням знищила б сигнал. Краще кодувати його окремою категорією або прапорцем `is_missing`. Можливий конфаундер — період: якщо невідомі статуси зосереджені у 2008 році з низькою конверсією (Q8), частина ефекту пояснюється часом, і це варто перевірити окремо.

### Запит 5

**Business-question:** У скількох клієнтів статус іпотеки відрізняється від статусу споживчого кредиту, з урахуванням невідомих значень?

In [ ]:
q5 = q("""
SELECT
    COUNT(*) FILTER (WHERE has_housing_loan <> has_personal_loan)                   AS differ_by_neq,
    COUNT(*) FILTER (WHERE has_housing_loan IS DISTINCT FROM has_personal_loan)     AS differ_by_is_distinct,
    COUNT(*) FILTER (WHERE has_housing_loan = has_personal_loan)                    AS same_by_eq,
    COUNT(*) FILTER (WHERE has_housing_loan IS NOT DISTINCT FROM has_personal_loan) AS same_null_safe,
    COUNT(*) FILTER (WHERE has_housing_loan IS NULL AND has_personal_loan IS NULL)  AS both_null,
    COUNT(*) AS total_rows
FROM hw3_bank_contacts
""")
q5

,differ_by_neq,differ_by_is_distinct,same_by_eq,same_null_safe,both_null,total_rows
0,20442,20442,19756,20746,990,41188


**Interpretation:** У **20 442** клієнтів іпотека та споживчий кредит мають різні статуси, і тут `<>` та `IS DISTINCT FROM` дають однаковий результат. Причина в тому, що `NULL` в обох колонках завжди трапляється **разом** (ті самі 990 рядків), тож пар «відоме vs NULL» немає. Різниця видна на рівності: `=` знаходить **19 756** збігів, а `IS NOT DISTINCT FROM` — **20 746**, бо додатково вважає «однаковими» 990 рядків з `NULL = NULL`. Для порівнянь nullable-колонок (дедуплікація, CDC, join) потрібні null-safe оператори, інакше рядки з пропусками мовчки випадають (як у self-join у 3.2).

### Запит 6

**Business-question:** Яка конверсія у «теплого» сегмента: клієнтів на мобільному, які мали успішний попередній контакт або нещодавній повторний дзвінок і не мають підтвердженого дефолту?

In [ ]:
q6 = q("""
SELECT
    contact_channel,
    prev_outcome,
    COUNT(*) AS n_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
WHERE contact_channel = 'cellular'
  AND (
        prev_outcome = 'success'
        OR (days_since_prev_contact IS NOT NULL AND days_since_prev_contact <= 7)
      )
  AND NOT COALESCE(has_credit_default, FALSE)
GROUP BY contact_channel, prev_outcome
ORDER BY n_contacts DESC
""")
q6

,contact_channel,prev_outcome,n_contacts,pct_subscribed
0,cellular,success,1270,65.20
1,cellular,failure,46,47.83


**Interpretation:** «Теплий» сегмент дуже малий — 1 316 контактів (3,2 % набору), — але конверсія в ньому в 4–6 разів вища за базову. Клієнти на мобільному з **успішною** попередньою кампанією підписуються в **65,20 %** випадків (1 270 контактів), а з невдалою кампанією, але свіжим (≤ 7 днів) повторним контактом — у **47,83 %** (лише 46 рядків). Звідси практичний висновок для кампанії: пріоритизувати повторні дзвінки клієнтам з позитивною історією.

### Запит 7

**Business-question:** Чи є макроекономічні показники атрибутом клієнта, чи контекстом місяця (скільки унікальних комбінацій вони дають)?

In [ ]:
q7 = q("""
SELECT DISTINCT
    contact_month,
    emp_var_rate,
    cons_price_idx,
    cons_conf_idx,
    nr_employed
FROM hw3_bank_contacts
ORDER BY contact_month
LIMIT 12
""")
q7

,contact_month,emp_var_rate,cons_price_idx,cons_conf_idx,nr_employed
0,2008-05-01,1.1,93.994,-36.4,5191.0
1,2008-06-01,1.4,94.465,-41.8,5228.1
2,2008-07-01,1.4,93.918,-42.7,5228.1
3,2008-08-01,1.4,93.444,-36.1,5228.1
4,2008-10-01,-0.1,93.798,-40.4,5195.8
5,2008-11-01,-0.1,93.200,-42.0,5195.8
6,2008-12-01,-0.2,92.756,-45.9,5176.3
7,2009-03-01,-1.8,92.843,-50.0,5099.1
8,2009-04-01,-1.8,93.075,-47.1,5099.1
9,2009-05-01,-1.8,92.893,-46.2,5099.1


**Interpretation:** `SELECT DISTINCT` показує, що п'ять макропоказників разом дають лише **26 унікальних комбінацій** на 41 188 рядків — рівно по одній на місяць (див. 2.4). Значить, це **контекст місяця, а не атрибут клієнта**. `emp_var_rate` і `nr_employed` до того ж змінюються лише щокварталу. Для моделі вони фактично кодують період, тож при випадковому split модель може «запам'ятати» місяці з високою конверсією замість того, щоб узагальнювати. Це аргумент за split за часом.

### Запит 8

**Business-question:** Як змінювались обсяг дзвінків і конверсія з часом (за роками кампаній)?

In [ ]:
q8 = q("""
SELECT
    EXTRACT(YEAR FROM contact_month)::INT AS contact_year,
    COUNT(*) AS n_contacts,
    COUNT(DISTINCT contact_month) AS n_months,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed,
    ROUND(AVG(euribor3m), 3) AS avg_euribor3m
FROM hw3_bank_contacts
GROUP BY contact_year
ORDER BY contact_year
""")
q8

,contact_year,n_contacts,n_months,pct_subscribed,avg_euribor3m
0,2008,27690,7,4.84,4.817
1,2009,11440,10,19.48,1.232
2,2010,2058,9,52.14,0.813


**Interpretation:** Розподіл у часі різко нерівномірний. У **2008** році — 27 690 дзвінків (67 % набору) з конверсією лише **4,84 %**, у **2010** — 2 058 дзвінків з конверсією **52,14 %**. Одночасно Euribor впав з 4,82 % до 0,81 %. Банк, імовірно, перейшов від масових холодних дзвінків до таргетованих (менше дзвінків, «тепліша» база). Це сильний **concept/population drift**: модель, навчена на всьому періоді, переоцінить конверсію для масових кампаній. Тому валідацію варто робити out-of-time (наприклад, train 2008–2009, test 2010).

### Запит 9

**Business-question:** Чи дають результат повторні дзвінки одному клієнту в межах кампанії?

In [ ]:
q9 = q("""
SELECT
    LEAST(campaign_contacts, 10) AS campaign_contacts_capped,
    COUNT(*) AS n_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed
FROM hw3_bank_contacts
GROUP BY campaign_contacts_capped
ORDER BY campaign_contacts_capped
""")
q9

,campaign_contacts_capped,n_contacts,pct_subscribed
0,1,17642,13.04
1,2,10570,11.46
2,3,5341,10.75
3,4,2651,9.39
4,5,1599,7.50
5,6,979,7.66
6,7,629,6.04
7,8,400,4.25
8,9,283,6.01
9,10,1094,3.56


**Interpretation:** Конверсія монотонно спадає з кожним додатковим дзвінком: **13,04 %** для першого, 9,39 % для четвертого і **3,56 %** для 10+ дзвінків (група з `LEAST(..., 10)`). **88 %** усіх підписок (4 085 з 4 640) припадає на клієнтів, яким дзвонили не більше 3 разів, далі віддача мізерна. Тож бізнес-пропозиція — ліміт ~5 дзвінків на клієнта. Для ML важливо, що `campaign` включає й **останній** дзвінок. Як ознаку на момент прогнозу можна використати лише кількість дзвінків **до** поточного.

### Запит 10

**Business-question:** Як результат попередньої кампанії пов'язаний з успіхом поточної?

In [ ]:
q10 = q("""
SELECT
    prev_outcome,
    COUNT(*) AS n_contacts,
    ROUND(AVG(subscribed::INT) * 100, 2) AS pct_subscribed,
    ROUND(AVG(days_since_prev_contact), 1) AS avg_days_since_prev
FROM hw3_bank_contacts
GROUP BY prev_outcome
ORDER BY pct_subscribed DESC
""")
q10

,prev_outcome,n_contacts,pct_subscribed,avg_days_since_prev
0,success,1373,65.11,5.6
1,failure,4252,14.23,10.1
2,nonexistent,35563,8.83,NaN


**Interpretation:** Результат попередньої кампанії — найсильніший сигнал серед пре-контактних ознак. Після **success** конверсія становить **65,11 %**, після **failure** — 14,23 %, без попередньої кампанії (`nonexistent`, 86 % бази) — лише **8,83 %**. Успішні клієнти ще й мають свіжіший попередній контакт: у середньому 5,6 дня проти 10,1 після невдачі. `NaN` для `nonexistent` — коректний результат, бо `AVG` по самих `NULL` дає `NULL`. Це підтверджує рішення не зберігати `999` як число: інакше середнє для failure було б **966** днів.

### Запит 11

**Business-question:** Наскільки тривалість дзвінка відрізняється між успішними та неуспішними контактами (тобто чи є duration leakage)?

In [ ]:
q11 = q("""
SELECT
    subscribed,
    COUNT(*) AS n_contacts,
    ROUND(AVG(call_duration_sec), 1) AS avg_duration_sec,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY call_duration_sec) AS median_duration_sec,
    COUNT(*) FILTER (WHERE call_duration_sec < 60) AS calls_under_1_min,
    ROUND(COUNT(*) FILTER (WHERE call_duration_sec < 60) * 100.0 / NULLIF(COUNT(*), 0), 2)
        AS pct_under_1_min
FROM hw3_bank_contacts
GROUP BY subscribed
ORDER BY subscribed
""")
q11

,subscribed,n_contacts,avg_duration_sec,median_duration_sec,calls_under_1_min,pct_under_1_min
0,False,36548,220.8,163.5,4180,11.44
1,True,4640,553.2,449.0,1,0.02


**Interpretation:** Медіанний дзвінок з підпискою триває **449 с**, без підписки — **163,5 с** (середні 553 і 221). Серед успішних контактів лише **1** дзвінок коротший за хвилину (0,02 %), серед неуспішних таких 11,44 %. `duration` майже ідеально відділяє класи, бо **є наслідком** результату: довга розмова відбувається, коли клієнт уже погоджується. Значення стає відомим лише після завершення дзвінка, тому для моделі «кому дзвонити» це **leakage** і ознаку треба виключити. Її можна лишити тільки для бенчмарку чи аналізу після дзвінка.

### Покриття обов'язкових конструкцій

| Конструкція | Де використано |
|---|---|
| явний перелік колонок у `SELECT` | усі запити (без `SELECT *`) |
| `WHERE` з `AND / OR / NOT` і дужками | Q6 |
| `IN` / `BETWEEN` | Q2 (`BETWEEN`); також `IN` в аудиті 3.3(в) |
| `LIKE` / `ILIKE` | Q3 |
| `IS NULL` / `IS NOT NULL` | Q4, Q6 |
| `COALESCE` / `NULLIF` | Q1, Q3, Q6 (`COALESCE`), Q11 (`NULLIF`) |
| `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` | Q5 (обидва) |
| `DISTINCT` | Q7 (`SELECT DISTINCT`), Q8 (`COUNT(DISTINCT)`) |
| `ORDER BY` + `LIMIT` | Q1, Q7 |
| `GROUP BY` (≥ 2 запити) | Q1, Q2, Q3, Q4, Q6, Q8, Q9, Q10, Q11 |

## 6. Reflection

**Проблеми якості.** SQL `NULL` у файлі немає. Пропуски закодовано маркером `unknown` у шести категоріях, а найбільше їх у `default` (20,9 %). Пропуски систематичні: `housing` і `loan` невідомі в одних і тих самих рядках, і група з невідомими статусами конвертується вдвічі гірше. Окрема проблема — перевантажений sentinel: `pdays = 999` позначає і «не було контакту», і, у 4 110 рядках з `poutcome = failure`, щось інше. Також знайдено 12 точних дублікатів, нульові дзвінки і відсутність року, який довелося відновлювати з порядку рядків.

**Features.** Як ознаки я використала б профіль клієнта (`age`, `job`, `education`, `marital`), кредитні статуси з явною категорією «невідомо», канал, `previous_contacts` і `prev_outcome`. Не використовувала б `duration` (leakage), технічні `contact_id`/`src_line_no`/`loaded_at` і `default` як окрему бінарну ознаку (лише 3 «yes»). Макроіндикатори дозволені лише за умови out-of-time валідації, бо вони кодують місяць.

**Leakage.** Головний ризик — `duration`, відомий лише після дзвінка. `campaign` включає поточний дзвінок, тож коректна ознака — кількість дзвінків **до** нього. Випадковий split змішує періоди з конверсією 5 % і 52 %, тому потрібен split за часом.

**Пропозиції для pipeline.** Віддавати повну дату й час дзвінка (`TIMESTAMPTZ`) і стабільний ID клієнта та контакту. Розвести семантики через окремі колонки замість sentinel `999`. Кодувати пропуски як `NULL` з окремим `missing_reason`. Додати контракт схеми з перевірками на дублікати.

**ML-задача.** Бінарна класифікація (`subscribed`) з сильним дисбалансом класів. Її можна подати і як ranking — пріоритизацію списку дзвінків за ймовірністю успіху, — що природно для телемаркетингу.